# PID Controller Simulation

## Learning Objectives

After completing this exercise, the student will be able to:
- Create a simple model for a PID controller
- Numerically integrate the motion of a moving body
- Implement and analyze the effects of an integral and derivative term on the system response
- Explore how a more complex system affects the PID controller's behavior

## Introduction
Small drones are now a part of everything from warehouse automation to aerial photography. Keeping them stable in the air is crucial to their success. Suppose we want to keep one exactly 10 meters above the ground. This seems easy — just apply enough thrust to counteract gravity, right?

But how do we know how much thrust to apply? And what if the environment isn't still? A gust of wind could push the drone up or down, and we need to adjust the thrust accordingly. How do we do that smoothly, and without the drone oscillating up and down?

This is a common problem in <a href="https://en.wikipedia.org/wiki/Control_theory">Control theory</a>. To keep the drone flying steadily at the right height, we need a way to adjust how much thrust force to apply. One method for doing this is by using a **PID controller**. PID stands for "Proportional, Integral, and Derivative." It is a type of feedback controller whose output is determined by comparing a measured value to it's target value.  In the case of a drone, the controller output is the magnitude of thrust, which is determined by comparing the actual height of the drone to some target height that we wish to maintain.

The controller output is based on three considerations:
1. **Proportional (P)**: How far is the drone from the desired height right now? If it is too low, increase thrust. If it is too high, reduce thrust.
2. **Integral (I)**: Has the drone been off target for a while? If it has been too low for a long time, gradually increase thrust to correct for that. This helps remove any steady offset.
3. **Derivative (D)**: How fast is the height changing? If the drone is rising quickly, reduce thrust to prevent overshooting. If it is falling quickly, increase the thrust to slow the descent.
                       
The only things the PID controller needs to know are the true height of the drone and the target height. It continuously compares the two to calculate the difference between them, known as the **error**, and then uses the P, I, and D components to decide how much thrust to apply.

In this project, we will simulate a one-dimensional PID controller that attempts to keep a drone stable at a desired height. We will explore how the different components of the PID controller affect the response and see how a seemingly simple feedback loop can yield powerful control behavior.

Before we can simulate the PID controller, we need to understand the forces acting on the drone. In the simplest scenario, there are two forces acting on the drone in the vertical direction, the drone's thrust (upward) and gravity (downward). According to Newton's third law, the drone experiences no acceleration when these forces are balanced.  If the thrust exceeds gravity, the drone will accelerate upwards.  If gravity exceeds the thrust, the drone will instead accelerate downwards.

## 2. The P Controller

We will start by implementing the proportional, P, component of the PID controller. The output $u(t)$ of this component is linearly proportional to the error,
$$
u(t) = K_p e(t),
$$
where $K_p$ is a proportionality constant and $e(t)$ is the difference between a target and measured observable. 

In our case, the error is the difference between the target height $y_{\mathrm{target}}$ and the measured height $y(t)$, or $e(t)=y_{\mathrm{target}}-y(t)$, and the output $u(t)$ represents the drone's thrust.  When the drone is far below the target height, the error is large, which makes the controller output (thrust) large, and the drone rises towards the target height. As the drone approaches the target height, the error becomes smaller, which also reduces the amount of thrust applied. 

In this section, we will see how a proportional controller alone affects the drone's motion, and why it is often not enough on its own. 


### a) Implementing the acceleration function

To simulate the drone's motion, we need to calculate the vertical acceleration. This is done by finding the net force acting on the drone, which is the difference between the thrust and the gravitational force, and applying Newton's third law. 

The gravitational force is constant and can be represented as $F_g = mg$, where $m$ is the mass of the drone and $g$ is the acceleration due to gravity. Using Newton's second law, $F_{\mathrm{net}}=ma$, we see that the drone's acceleration $a(t)$ as a function of the thrust force $u(t)$ and the gravitational force $F_g$ is given by
$$ a(t) = \frac{u(t) - F_g}{m} = \frac{u(t)}{m} - g,$$
where we have taken upwards to be the positive $y$ direction.

Based on this expression, implement a function `acceleration(thrust, m, g)` that calculates the vertical acceleration of the drone based on its thrust, mass, and $g$.

### b) The proportional component

As you can see, the acceleration depends on the thrust force $u(t)$, which we have not yet defined.  Write a function `proportional_component(K_P, target, measured)` which calculates the thrust $u(t)$ for a given value of the proportionality constant $K_P$ as well as the target and measured heights.

### c) Simulate the P controller

Now that we have all the pieces, we are ready to simulate the movement of the drone using the proportional controller.

In the following code section, you will implement the Euler-Cromer method to numerically integrate the velocity and position of the drone. The Euler-Cromer method is an extension of the Euler method and is significantly more stable and reliable for simulating physical systems when the acceleration is known.

The Euler-Cromer equations for this system are:

\begin{align}
    v(t + \Delta t) &= v(t) + a(t) \Delta t\\
    y(t + \Delta t) &= y(t) + v(t + \Delta t) \Delta t
\end{align}
where $v(t)$ is the velocity at time $t$ and $\Delta t$ is the time step.

Assume that the drone (with a mass of 1 kg) is initially at rest on the ground and we would like it to maintain a height of 10 m.  Use your function `proportional_component(K_P, target, measured)` to calculate the initial thrust, then use `acceleration(thrust, m, g)` to calculate the resulting acceleration, and finally use the above equations to calculate the velocity and position of the drone at the next time step.  By iteratively applying this method, you can determine the height of the drone $y(t)$ for all time steps.

Experiment with the value of $K_p$ until you find something resonable.  Can you tune this constant so that the drone maintains the target height?

Your results may not be great, but that is expected. The P controller is not very good at keeping the drone stable.  This should not be too surprising.  The P term represents a linear restoring force, like a simple harmonic oscillator, so we could expect similar behavior here.

The next step is to add the derivative component to the controller, to see if this improves the situation.

## 3) PD controller
We have just witnessed the limitations of a P controller. It only reacts to the current error, meaning the difference between the target and the measurement heights, without considering how the error is changing over time. This often leads to overshooting and oscillations in the system.

To improve the controller, we introduce a derivative component, D, which responds to the rate of change of the error. This allows the controller to anticipate where the system is heading, not just where it currently is. In other words, the derivative component acts as anticipatory control, pushing back harder when the error is changing rapidly and backing off when things are stabilizing. It effectively adds damping, which helps smooth out the response and prevent overshooting.

Mathematically, the controller output is now proportional to both the error and its derivative:
$$u(t) = K_p e(t) + K_d \frac{\mathrm{d}e(t)}{\mathrm{d} t},$$ where $K_d$ is another proportionality constant.

### a) Understanding the derivative component
Before we implement the derivative component, let us first try to understand how it affects the system. The derivative component is based on the rate of change of the error, which can be approximated using the backward difference method as
$$\frac{\mathrm{d} e(t)}{\mathrm{d} t} \approx \frac{e(t) - e(t - \Delta t)}{\Delta t},$$
where $e(t - \Delta t)$ is the error at the previous time step.

Now consider these cases:
1. What happens to the thrust force $u(t)$ when the drone is **below** the target height and **rising**?
2. What happens to the thrust force $u(t)$ when the drone is **above** the target height and **falling**?
3. What happens with the thrust force $u(t)$ when the drone is at a **stable** height (not necessarily the target height)?

In each case, give the sign of the P and D components, and specify the effect of the D component on the drone's motion compared to a pure P controller. *Hint: Think about the sign of the error and the sign of its derivative in each case*

### b) Implementing the derivative component

The next thing we need to do is to implement the derivative of the error into the simulation using the backwards difference approximation.

Write a function `derivative_component(K_D, current_error, previous_error, dt)` that takes as arguments the proportionality constant $K_d$, the current error, the error during the previous time step, and the size of the time step, which calculates the derivative component.

### c) Simulating the PD controller
Now that you have implemented the derivative component, you are ready to simulate the movement of the drone using the PD controller.

Use the Euler Cromer method to numerically integrate the velocity and position of the drone. The PD controller will use the proportional and derivative components to calculate the thrust force. Also plot the result and observe how it changes compared to the P controller.  Can you tune the proportionality constants so that the drone maintains the target height?

Hopefully, you have now seen an improvement in the stability of the drone. The PD controller is much better at keeping the drone stable than when we only had the P controller. But we can still see that we are not quite there yet. Again, do not worry, we will improve the controller once again by adding the integral term.

## 4) PID controller
We have now seen the effects of the P and D terms in the controller. While the derivative term helps by anticipating changes in error and damping the system response, the PD controller is still not perfect—particularly when it comes to eliminating steady-state errors, where the drone hovers at a height that is not exactly the desired height.

To address this problem, we will introduce the integral term, I. The integral term accumulates the error over time, effectively "remembering" past errors. If there is any persistent difference between the target and the measurement, the integral term will continue to grow, pushing the controller to reduce the error. As the error decreases, the contribution from the proportional term naturally weakens, but the integral term, having built up over time, continues to drive the output until the error is zero. Once the error reaches zero, the integral term stops growing, stabilizing the system. This cumulative effect is what allows the controller to eliminate residual or static errors that the proportional and derivative terms alone cannot remedy.

Together, the three terms combine into a PID controller, which is defined as follows:

$$u(t) = K_P e(t) + K_D \frac{\mathrm{d} e(t)}{\mathrm{d} t} + K_I \int_0^t e(t) dt $$

### a) Implementing the integral component

We want to implement the integral of the error into the simulation by using a simple numerical integration method, such as a Riemann sum:
$$\int_0^t e(t) dt \approx \sum_{i=0}^{n} e(t_i) \Delta t$$
The Riemann sum will be accumulated in a variable, which needs to be initialized as zero, and then updated in each iteration of the simulation loop.

Write a function `integral_component(K_I, error_integral)` that takes the proportionality constant $K_I$ and the integral of the error from zero to the current time, which calculates the integral component.

### b) Simulating the PID controller

Now that you have implemented all the necessary functions for the PID controller, you should be able to simulate the drone’s movement using the PID controller combined with the Euler Cromer method.  Once again, plot the results.

Hopefully, you have now seen a significant improvement in the drone’s stability. Your PID controller should be able to keep the drone steady at the target height.

## 5) Include air resistance

At the low speeds considered here, air resistance (or drag) is approximately linear.  That is, $F_d = -kv$, where $F_d$ is the resulting drag force, $k$ is the drag coefficient, and $v$ is the velocity.  The minus sign here indicates that the direction of the force opposes the direction of the velocity.

Write a function that takes the velocity and drag coefficient as arguments, and calculates the resulting drag force.

Rewrite the `acceleration` function that you defined above, to also include the affect of wind resistance.

Simulate the drone’s movement once again using the PID controller. Use the same proportionality constants as in the previous simulation, and set the drag coefficient to a similar value. Plot the results.

How does wind resistance affect the system?

## 6) Include random wind gusts

In the real world, there are often random effects that can affect the stability of the drone. For example, wind gusts can push the drone up or down, and we need to adjust the thrust accordingly. In this exercise, we will add a random effect to the simulation to see how the PID controller reacts to it.

The following function takes the wind amplitude as argument, and returns a random gust of wind (with magnitude less than the specified amplitude, but in a random direction).

In [1]:
def wind_force(wind_amplitude):
    """
    Randomly generates a wind
    """
    return np.random.uniform(-wind_amplitude, wind_amplitude)

Rewrite the acceleration function that you defined above, to also include the effect of random wind gusts at each time step.

Simulate the motion of the drone, now including random wind gusts, and plot the results.

## 7) Moving target

Finally, let's try to maintain a time-varying target height.  Define a function `moving_target(t)`, which takes the time as argument, and calculates the target height given by $10 + [5\sin(t) \cdot e^{-0.09t}]$.

Plot the moving target as a function of time.

Simulate the motion of the drone as it attempts to maintain this time-varying target height and plot the results.